# Retrieval Augmented Generation
In this notebook, we'll showcase how to build an AI agent that implements retrieval augmented generation powered by a Vector Database. This will allow you to feed external data sources (particularly large documents that would exceed LLM context windows) into your agent and get responses grounded on your ingested data sources (the source of truth)

In [7]:
import os
import bs4
import requests
import langsmith
from dotenv import load_dotenv

from langchain.tools import Tool
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader, PyPDFLoader

In [8]:
load_dotenv()

# LLM API configuration
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

# LangSmith configuration
LANGSMITH_TRACING = os.getenv("LANGSMITH_TRACING", "true")
LANGSMITH_API_KEY = os.getenv("LANGSMITH_API_KEY")
LANGSMITH_PROJECT = os.getenv("LANGSMITH_PROJECT", "default")
LANGSMITH_WORKSPACE_ID = os.getenv("LANGSMITH_WORKSPACE_ID")

## Building our Retrieval Source
We'll create a RAG system that uses the Kenyan and Tanzanian constitution as the retrieval sources.

### 1/ Embedding Model & Vector Database
- Instantiate our **embedding model** to convert our text into a vector space; 
- and our **vector store** where we'll store our vector embeddings.

In [32]:
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001") # Embeddings model
vector_store = InMemoryVectorStore(embeddings) # In-memory vector store to hold document embeddings

### 2/ Document Loaders & Chunking
Load data from our actual sources. In this tutorial we'll only covered 3 types of documents:
- PDF-based Documents
- Web-based Documents
- CSV-based Documents (Recursive Loading)

Once a document is loaded, we'll split it into chunks and load every chunk into our vector store.

#### 2.1. Loading PDF Files
Here we'll pass the Kenyan constitution as our first document, available in PDF format.

In [14]:
# Load a PDF document
loader = PyPDFLoader("../sources/kenyan-constitution.pdf")
docs = loader.load()

print(f'''Number of docs: {len(docs)}. Each doc is a page from the PDF. 
      Now, let's see some sample content from the first 5 docs:''')

for i in range(5):
    print(f"\n--- Doc {i+1} ---")
    print(docs[i].page_content)

Number of docs: 193. Each doc is a page from the PDF. 
      Now, let's see some sample content from the first 5 docs:

--- Doc 1 ---
LAWS OF KENYA
THE CONSTITUTION OF KENYA, 2010
Published by the National Council for Law Reporting
with the Authority of the Attorney-General
www.kenyalaw.org

--- Doc 2 ---
Constitution of Kenya, 2010
THE CONSTITUTION OF KENYA, 2010
ARRANGEMENT OF ARTICLES
PREAMBLE
CHAPTER ONE—SOVEREIGNTY OF THE PEOPLE AND  
SUPREMACY OF THIS CONSTITUTION
1—Sovereignty of the people.
2—Supremacy of this Constitution.
3—Defence of this Constitution.
CHAPTER TWO—THE REPUBLIC
4—Declaration of the Republic.
5—Territory of Kenya.
6—Devolution and access to services.
7—National, official and other languages.
8—State and religion.
9—National symbols and national days.
10—National values and principles of governance.
11—Culture.
CHAPTER THREE—CITIZENSHIP
12—Entitlements of citizens.
13—Retention and acquisition of citizenship.
14—Citizenship by birth.
15—Citizenship by registrat

When loading data from multiple different sources/files and with different context areas, it is important encode metadata into your docs before the vector embeddings. 

In [22]:
## Let's see the metadata of the first document
docs[0].metadata

{'producer': 'LibreOffice 3.6',
 'creator': 'Writer',
 'creationdate': '2013-02-28T17:50:04+03:00',
 'title': 'Rev. 2010]',
 'author': 'Andare Jeffrey',
 'source': '../sources/kenyan-constitution.pdf',
 'total_pages': 193,
 'page': 0,
 'page_label': '1'}

In [40]:
# Add custom metadata and update existing metadata
for i, doc in enumerate(docs):
    # Add custom metadata fields
    doc.metadata.update({
        'title': 'Kenyan Constitution PDF',
        'category': 'Constitution',
        'country': 'Kenya',
    })
    # Drop unnecessary metadata fields if they exist
    doc.metadata.pop('producer', None)  # Remove 'producer' field if it exists
    doc.metadata.pop('creationdate', None)  # Remove 'creation_date' field if it exists
    doc.metadata.pop('creator', None)  # Remove 'creator' field if it exists
    doc.metadata.pop('author', None)  # Remove 'author' field if it exists

print(f"Updated metadata for {len(docs)} documents")
print(f"Sample metadata from first document: {docs[0].metadata}")

Updated metadata for 193 documents
Sample metadata from first document: {'title': 'Kenyan Constitution PDF', 'source': '../sources/kenyan-constitution.pdf', 'total_pages': 193, 'page': 0, 'page_label': '1', 'document': 'Kenyan Constitution PDF', 'category': 'Constitution', 'country': 'Kenya'}


Now, we can split the documents into chunks, embed them and add them into the vector store

In [ ]:
## Split documents into smaller chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,  # chunk size (characters)
    chunk_overlap=500,  # chunk overlap (characters)
    add_start_index=True,  # track index in original document
)
all_splits = text_splitter.split_documents(docs)

print(f"Split PDF into {len(all_splits)} sub-documents.")

Split blog post into 355 sub-documents.


In [42]:
## Embed and add documents to the vector store
document_ids = vector_store.add_documents(documents=all_splits)
print(document_ids)

['35f498e0-1578-4a44-b065-3b3833a0dcf2', '34983464-53a8-4ec3-ae9c-c05019629cea', '7b89000f-8836-471a-8e4a-f66295ff353e', 'df3d15c4-f442-466d-8c84-354b11dcc003', 'b97025f8-4533-4e0f-a019-48e48b01abc1', '7c3a56ac-ffb6-4d8f-bee3-be158e6d7dfb', '3dfd00ff-2c95-4425-a01b-a0fe48f65688', '9d73ef6c-1cd7-4eba-a2c6-38f58f9c0c79', '75a0fe99-79fe-4b90-abdb-5f5906e21ef8', '5d7c97fa-478d-4e8a-b460-92ba700f5136', '6c245a38-05a2-4472-8e3f-c1ad30197976', '3976a56f-cb1b-46b9-bf92-5515ca0257f5', '86c1f8ff-8abc-4f7a-ac05-0406315ce66c', '72da8c04-b8dd-44da-8513-390bab8f8458', 'b6827578-e502-4f4e-94f5-0bd51c6c9daf', 'c1239d18-ab5a-4fa7-9889-ebe9546f4c97', '22dd2140-5d7f-4fd3-9a3a-f796991e73b8', 'f4763235-7eae-4f3e-8154-516367ec156b', '39f570b1-a878-4f6a-ae35-55a891112e5b', '24b9cdb8-a48f-4cbc-a4ad-1ac3ec302aa8', '770ab5a4-1606-418f-b7f5-92fb1ead0e7e', '4f1aa161-9cf6-437f-876e-03dfeb34bb98', 'f60fe679-cee5-4de0-b2a8-6ae5612d92b1', '3db61d59-19dc-469c-ab0a-afe9e1f5def3', '628a240a-3b64-4a1b-aa06-e8fa8cd57ab2',

#### 2.2. Loading Web-Based Files
Below we'll pass the Tanzanian constitution as our second document. 

In [43]:
# Load web-based document - Tanzanian Constitution
web_loader = WebBaseLoader("https://www.constituteproject.org/constitution/Tanzania_2005")
web_docs = web_loader.load()

print(f"Number of web docs: {len(web_docs)}")
#print(f"Sample content from first web document:\n{web_docs[0].page_content[:500]}")

Number of web docs: 1


Let us encode our metadata as we did before. We check what metadata is available, add missing metadata, then drop all irrelevant ones.

In [44]:
web_docs[0].metadata

{'source': 'https://www.constituteproject.org/constitution/Tanzania_2005',
 'title': 'Tanzania (United Republic of) 1977 (rev. 2005) Constitution - Constitute',
 'description': "Tanzania (United Republic of)'s Constitution of 1977 with Amendments through 2005",
 'language': 'en'}

In [45]:
# Add custom metadata to web documents
for i, doc in enumerate(web_docs):
    # Add custom metadata fields
    doc.metadata.update({
        'title': 'Tanzanian Constitution Web Document',
        'category': 'Constitution',
        'country': 'Tanzania',
    })
    doc.metadata.pop('description', None)  # Remove 'description' field if it exists
    doc.metadata.pop('language', None)  # Remove 'language' field if it exists

print(f"Updated metadata for {len(web_docs)} web documents")
print(f"Sample metadata from first web document: {web_docs[0].metadata}")

Updated metadata for 1 web documents
Sample metadata from first web document: {'source': 'https://www.constituteproject.org/constitution/Tanzania_2005', 'title': 'Tanzanian Constitution Web Document', 'category': 'Constitution', 'country': 'Tanzania'}


Now, we can split the document into chunks then add it to the vector database.

In [46]:
# Split documents into chunks
## Split documents into smaller chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,  # chunk size (characters)
    chunk_overlap=500,  # chunk overlap (characters)
    add_start_index=True,  # track index in original document
)
all_splits = text_splitter.split_documents(web_docs)

print(f"Split web document into {len(all_splits)} sub-documents.")

Split web document into 220 sub-documents.


In [47]:
## Embed and add documents to the vector store
document_ids = vector_store.add_documents(documents=all_splits)
print(document_ids)

['80835bb7-d19c-46ab-b96a-af90f2ca1704', 'dca2267e-7eae-41ab-8b89-f194277357af', '2550ae26-48dc-415a-ac6d-8da0a82a3304', '5c8c9d29-aa8d-43aa-8472-8ec6e63063e1', '6fd805eb-f725-4990-97ff-2d595c0f1fee', '504c5193-3df7-4484-ae04-44bde47232a6', '6b1a2b5c-9d45-4428-ab8c-88e5c3ea283b', '1bd1ddce-ecf7-4a5e-9552-42cca8dea603', '7be37556-5cac-440f-b859-bbbaa62796d2', 'cc2453fa-9008-47c4-9313-82c2a7567c45', '38fcd90f-4688-4521-9ff5-d6ef9a0ae653', '23f07c22-4788-4c0f-99b7-ef3e2c86216d', '7660dbc2-6c9a-43ba-9fc5-b9b5f0335732', '29037d11-2ead-440d-bca2-8d11641f27cc', '63f51ce0-49e6-4f6f-a82d-0556d54c6d93', 'df9cf947-cd5d-4f4f-9f41-534bbe877c1c', '074cdb8c-eb2f-46c3-9a3c-ed9959e5df78', 'ea9e9d41-a84d-4d36-836c-046b855f7011', 'af5e8a48-5432-4396-bd08-56633b267f59', '8eeefb64-5a34-4efb-ae15-59612a21e8c6', 'f960d6b5-4ac9-4780-8843-a6630f029ba6', '0ca5b5fe-bf99-462e-87bc-40f6b3ee9fa5', '32877f3e-6709-4b83-bd7e-9dedbe57d716', '2d5a215b-d6a0-430f-847a-61615d17fa57', 'db5cb779-29b9-481d-966a-4ab1e42a18ff',

#### Loading CSV-based Files
Coming soon: load documents recursively from a CSV containing multiple file urls.
- We'll scrape for constitution ammendments and Acts of Parliament and add urls/pdf files to a CSV
- We'll recursively load these files from a CSV document and embed them into our vector store

## Generate AI Responses grounded on our RAG Sources

### Instantiate the LLM

In [17]:
model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=5,
)

### Retrieve and Generate Response

In [48]:
def retrieve_and_generate(query: str, k: int = 5):
    # Retrieve relevant documents
    results = vector_store.similarity_search(query, k=k)
    
    # Create context from retrieved documents
    context = "\n\n".join([doc.page_content for doc in results])
    
    # Create messages with context
    messages = [
        SystemMessage(content=f'''You are an expert assistant. Use the provided context to answer the user's question accurately and comprehensively. 
        If the answer cannot be found in the context, say so clearly.
        
        Context:
        {context}'''),
        HumanMessage(content=query)
    ]
    
    # Generate response
    response = model.invoke(messages)
    return response.content

# Example usage
user_query = input("Enter your question about the documents: ")
answer = retrieve_and_generate(user_query)
print(f'''User Question: {user_query} \n Answer based on the documents:" \n {answer}''')

User Question: what happend after an election result is disputed in Kenya vs Tanzania 
 Answer based on the documents:" 
 Based on the provided context:

**In Kenya:**
*   Parliament is mandated to enact legislation to establish mechanisms for the timely settling of electoral disputes.
*   For elections *other than a presidential election*, petitions must be filed within twenty-eight days after the Independent Electoral and Boundaries Commission (IEBC) declares the election results.
*   Service of a petition can be done directly or by advertisement in a newspaper with national circulation.
*   The chairperson of the Independent Electoral and Boundaries Commission declares the result of the election and delivers a written notification to the Chief Justice and the incumbent President.

**Regarding Tanzania:**
The provided text does not contain any information about what happens after an election result is disputed in Tanzania. Therefore, I cannot answer that part of your question based o